# Building a Brain — Try it in PyTorch

This is an **optional** hands-on companion to [Chapter 3: Building a Brain](https://learnai.robennals.org/neurons). I'll build a single neuron from scratch, plot the sigmoid, hardcode AND/OR/NOT/NAND gates, watch a single neuron fail on XOR, train a two-layer network that succeeds, and stack into a deeper architecture.

*New to PyTorch? See the [PyTorch appendix](https://learnai.robennals.org/appendix-pytorch) for a beginner-friendly introduction.*

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

## The Building Block

A neuron does two things: a **weighted sum** of its inputs (plus a bias), then passes the result through an **activation function**. That's it. Let me build one by hand.

In [ ]:
def neuron(inputs, weights, bias, activation):
    weighted_sum = (inputs * weights).sum() + bias
    return activation(weighted_sum)

inputs = torch.tensor([0.5, 0.8])
weights = torch.tensor([0.7, -0.3])
bias = torch.tensor(0.1)

output = neuron(inputs, weights, bias, torch.sigmoid)
print(f'Inputs:  {inputs.tolist()}')
print(f'Weights: {weights.tolist()}, Bias: {bias.item()}')
print(f'Output:  {output.item():.4f}')

## Your First Neuron

The chapter's `NeuronFreePlayWidget` lets me poke at one neuron with two inputs and three knobs (two weights + bias). Let me try a few configurations and watch the output change.

In [ ]:
configs = [
    ('both weights positive, both inputs high', [1.0, 1.0], [2.0, 2.0], -1.0),
    ('one weight strongly negative',            [1.0, 1.0], [2.0, -3.0], 0.5),
    ('inputs zero, bias +2',                    [0.0, 0.0], [1.0, 1.0], 2.0),
    ('inputs zero, bias -2',                    [0.0, 0.0], [1.0, 1.0], -2.0),
]

for label, ins, ws, b in configs:
    inputs = torch.tensor(ins)
    weights = torch.tensor(ws)
    bias = torch.tensor(b)
    out = neuron(inputs, weights, bias, torch.sigmoid)
    print(f'{label}:')
    print(f'  inputs={ins}, weights={ws}, bias={b} -> output={out.item():.4f}')

## The Sigmoid

The activation function I'm using is the **sigmoid** — an S-shaped curve that smoothly squashes any number into the range 0 to 1. Large positive inputs become close to 1, large negative inputs close to 0, with a smooth transition through 0.5 at the middle.

In [ ]:
x = torch.linspace(-6, 6, 200)
y = torch.sigmoid(x)

plt.figure(figsize=(7, 4))
plt.plot(x.numpy(), y.numpy())
plt.axhline(0.5, color='gray', linewidth=0.5, linestyle=':')
plt.axvline(0,   color='gray', linewidth=0.5, linestyle=':')
plt.title('Sigmoid: smooth squash into (0, 1)')
plt.xlabel('input')
plt.ylabel('output')
plt.grid(True, alpha=0.3)
plt.show()

print(f'sigmoid(-100) = {torch.sigmoid(torch.tensor(-100.0)).item():.6f}')
print(f'sigmoid(   0) = {torch.sigmoid(torch.tensor(0.0)).item():.6f}')
print(f'sigmoid( 100) = {torch.sigmoid(torch.tensor(100.0)).item():.6f}')

## Sharper or Softer

If I scale all the weights and bias by a multiplier, the sigmoid steepens (toward a hard step) or flattens (toward a gentle slope). The boundary stays in the same place — only the sharpness changes. Very sharp neurons are harder to train (the slope is nearly zero on the flat parts), which is why real networks use regularization to keep weights moderate.

In [ ]:
x = torch.linspace(-6, 6, 200)

plt.figure(figsize=(7, 4))
for sharpness in [0.5, 1.0, 3.0, 10.0]:
    y = torch.sigmoid(x * sharpness)
    plt.plot(x.numpy(), y.numpy(), label=f'x {sharpness}')
plt.title('Sigmoid sharpness — multiplier on the input')
plt.xlabel('input')
plt.ylabel('output')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Neurons as Smooth Logic Gates

With the right weights, a single neuron can compute AND, OR, NOT, and NAND. Below I hardcode the weights for each gate and print its truth table — the same input combinations a real logic gate would handle. Large weights and biases push the sigmoid almost all the way to 0 or 1, so the truth tables come out crisp.

In [ ]:
gates = {
    'AND':  ([20.0, 20.0], -30.0),
    'OR':   ([20.0, 20.0], -10.0),
    'NAND': ([-20.0, -20.0], 30.0),
    'NOT':  ([-20.0],         10.0),
}

inputs_2 = [(0, 0), (0, 1), (1, 0), (1, 1)]
inputs_1 = [(0,), (1,)]

for gate, (ws, b) in gates.items():
    print(f'--- {gate} (weights={ws}, bias={b}) ---')
    weights = torch.tensor(ws)
    bias = torch.tensor(b)
    cases = inputs_1 if len(ws) == 1 else inputs_2
    for ins in cases:
        x = torch.tensor([float(v) for v in ins])
        out = neuron(x, weights, bias, torch.sigmoid)
        rounded = 1 if out.item() > 0.5 else 0
        print(f'  {ins} -> {out.item():.4f}  (rounded: {rounded})')
    print()

## A Single Neuron Can't Do XOR

**XOR** ("exclusive or") is true when exactly one of the inputs is true. The truth table is:

| A | B | XOR |
|---|---|-----|
| 0 | 0 |  0  |
| 0 | 1 |  1  |
| 1 | 0 |  1  |
| 1 | 1 |  0  |

No matter what weights I pick, a single neuron can't get all four cases right at once — this isn't a training failure, it's a *fundamental architectural limitation*. Let me train one and watch the loss plateau.

In [ ]:
# XOR training data
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = torch.tensor([[0.], [1.], [1.], [0.]])

torch.manual_seed(0)
single = nn.Sequential(
    nn.Linear(2, 1),
    nn.Sigmoid(),
)

loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(single.parameters(), lr=0.1)

losses = []
for epoch in range(2000):
    pred = single(X)
    loss = loss_fn(pred, y)
    losses.append(loss.item())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

plt.figure(figsize=(7, 3.5))
plt.plot(losses)
plt.title('Single-neuron XOR loss — plateaus, never reaches zero')
plt.xlabel('epoch')
plt.ylabel('BCE loss')
plt.grid(True, alpha=0.3)
plt.show()

print('Final predictions:')
with torch.no_grad():
    final = single(X)
    correct = 0
    for i in range(4):
        pred_val = final[i].item()
        pred_label = 1 if pred_val > 0.5 else 0
        target = int(y[i].item())
        ok = pred_label == target
        if ok: correct += 1
        mark = 'CORRECT' if ok else 'WRONG'
        print(f'  {X[i].tolist()} -> {pred_val:.4f} (predict {pred_label}, target {target})  {mark}')
    print(f'Accuracy: {correct}/4')

## Two Layers Solve XOR

Add one **hidden layer** with two neurons feeding into an output neuron. The two hidden neurons compute intermediate things (like "is at least one input on?" and "are they not both on?"), and the output neuron combines those into XOR. Same training loop as before — this time the loss drops to near zero and all four cases come out right.

In [ ]:
torch.manual_seed(1)
two_layer = nn.Sequential(
    nn.Linear(2, 2),
    nn.Sigmoid(),
    nn.Linear(2, 1),
    nn.Sigmoid(),
)

optimizer = torch.optim.Adam(two_layer.parameters(), lr=0.1)

losses = []
for epoch in range(5000):
    pred = two_layer(X)
    loss = loss_fn(pred, y)
    losses.append(loss.item())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

plt.figure(figsize=(7, 3.5))
plt.plot(losses)
plt.title('Two-layer XOR loss — drops to near zero')
plt.xlabel('epoch')
plt.ylabel('BCE loss')
plt.grid(True, alpha=0.3)
plt.show()

print('Final predictions:')
with torch.no_grad():
    final = two_layer(X)
    correct = 0
    for i in range(4):
        pred_val = final[i].item()
        pred_label = 1 if pred_val > 0.5 else 0
        target = int(y[i].item())
        ok = pred_label == target
        if ok: correct += 1
        mark = 'CORRECT' if ok else 'WRONG'
        print(f'  {X[i].tolist()} -> {pred_val:.4f} (predict {pred_label}, target {target})  {mark}')
    print(f'Accuracy: {correct}/4')

## Deeper Networks

Real networks stack many layers. PyTorch makes this easy with `nn.Sequential` — pass it a list of layers and it wires them up in order. Below I build a 4-input, 3-hidden-layer (8 neurons each), 1-output network. I'm not training it — just constructing it and running one forward pass to make depth tangible.

In [ ]:
deeper = nn.Sequential(
    nn.Linear(4, 8), nn.Sigmoid(),
    nn.Linear(8, 8), nn.Sigmoid(),
    nn.Linear(8, 8), nn.Sigmoid(),
    nn.Linear(8, 1), nn.Sigmoid(),
)

print(deeper)

n_params = sum(p.numel() for p in deeper.parameters())
print(f'\nTotal parameters: {n_params}')

# One forward pass with example inputs
example = torch.tensor([[0.5, -0.2, 1.1, 0.0]])
print(f'\nForward pass:')
print(f'  input shape:  {tuple(example.shape)}')
print(f'  output shape: {tuple(deeper(example).shape)}')
print(f'  output value: {deeper(example).item():.4f}')

---

*This notebook accompanies [Chapter 3: Building a Brain](https://learnai.robennals.org/neurons). Next up: [Chapter 4: Describing the World with Numbers](https://learnai.robennals.org/vectors) — where I'll see neurons as vector operations and discover why activation functions make depth meaningful.*

*New to PyTorch? See the [PyTorch appendix](https://learnai.robennals.org/appendix-pytorch) for a beginner-friendly introduction.*